<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/notebooks/fine_tune_with_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

~~~
Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# Notice of edits

This notebook has been edited by Team 207 of the 2026 488 CMPSC Project course for our own purposes. For the original notebook, please see [here](https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb).

# Fine-tune MedGemma with Hugging Face

This notebook demonstrates fine-tuning MedGemma on an image and text dataset for a vision task using Hugging Face libraries.

In this guide, you will use Hugging Face's [Transformer Reinforcement Learning (`TRL`)](https://github.com/huggingface/trl) library to train the model with Supervised Fine-Tuning (SFT), utilizing [Quantized Low-Rank Adaptation (QLoRA)](https://arxiv.org/abs/2305.14314) to reduce computational costs while maintaining high performance.


## Setup

To complete this tutorial, you'll need to have a runtime with sufficient resources to fine-tune the MedGemma model. **Note:** This guide requires a GPU that supports bfloat16 data type and has at least 40 GB of memory.

You can run this notebook in Google Colab using an A100 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **A100 GPU**.

### Get access to MedGemma

Before you get started, make sure that you have access to MedGemma models on Hugging Face:

1. If you don't already have a Hugging Face account, you can create one for free by clicking [here](https://huggingface.co/join).
2. Head over to the [MedGemma model page](https://huggingface.co/google/medgemma-4b-it) and accept the usage conditions.

### Configure your HF token

Generate a Hugging Face `write` access token by going to [settings](https://huggingface.co/settings/tokens). **Note:** Make sure that the token has write access to push the fine-tuned model to Hugging Face Hub.

If you are using Google Colab, add your access token to the Colab Secrets manager to securely store it. If not, proceed to run the cell below to authenticate with Hugging Face.

1. Open your Google Colab notebook and click on the 🔑 Secrets tab in the left panel. <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. Create a new secret with the name `HF_TOKEN`.
3. Copy/paste your token key into the Value input box of `HF_TOKEN`.
4. Toggle the button on the left to allow notebook access to the secret.

In [1]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

### Install dependencies

In [2]:
! pip install --upgrade --quiet bitsandbytes datasets evaluate peft tensorboard==2.19.0 transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 46.8 MB/s eta 0:00:00


### Download dataset

In [3]:
! wget -nc -q "https://drive.google.com/uc?export=download&id=1w0_XjEl4gCrDWxFnsJHrhsVEGTiF3Ayh" -O trainingData.zip
! unzip -q trainingData.zip

### Prepare raw dataset

In [4]:
import json
import glob
import os

data_directory = "/content/trainingFilesConverted/"

def load_drug_data(directory):
    all_data = []
    # Use glob to find all .json files in the directory
    file_paths = glob.glob(os.path.join(directory, "*.json"))

    if not file_paths:
        print(f"No JSON files found in {directory}. Check your path!")
        return None

    for path in file_paths:
        with open(path, 'r') as f:
            try:
                item = json.load(f)
                all_data.append(item)
            except Exception as e:
                print(f"Error loading {path}: {e}")

    return all_data

raw_json_list = load_drug_data(data_directory)

#### Example data point

Inspect a sample data point, which contains:

* `drug`: name of the drug in question
* `text`: collection of prescription information
* `interactions`: an array of interactions with the optional fields of: `@type`, `@precipitant`, `@precipitantCode`, and `@effect`

In [5]:
raw_json_list[0]

{'drug': 'TENEX',
 'text': 'CONTRAINDICATIONS \n\n Tenex is contraindicated in patients with known hypersensitivity to\n guanfacine hydrochloride.PRECAUTIONS \n\n General \n Like other antihypertensive agents, Tenex (guanfacine hydrochloride)\n should be used with caution in patients with severe coronary insufficiency,\n recent myocardial infarction, cerebrovascular disease, or chronic renal or\n hepatic failure. \n\n Sedation \n Tenex, like other orally active central\n α 2 -adrenergic agonists, causes sedation or drowsiness,\n especially when beginning therapy. These symptoms are dose-related (see\n\n ADVERSE REACTIONS \n ). When Tenex is\n used with other centrally active depressants (such as phenothiazines,\n barbiturates, or benzodiazepines), the potential for additive sedative effects\n should be considered. \n\n Rebound \n Abrupt cessation of therapy with orally active central\n α 2 -adrenergic agonists may be associated with increases\n (from depressed on-therapy levels) in pla

### Prepare conversational dataset

For this task, create a prompt and preprocess the data into a multimodal conversational format.

In [6]:
import json

def format_for_medgemma(raw_list):
    formatted_list = []
    for entry in raw_list:
        # Standardizing the instruction for extraction
        instruction = (
            f"You are a medical informatics expert. Extract all drug-drug interactions for the "
            f"drug '{entry['drug']}' from the following FDA label text. "
            f"Return the interactions as a structured JSON list."
        )

        # We format the labels for the Assistant to follow the exact JSON schema provided
        formatted_list.append({
            "messages": [
                {"role": "user", "content": f"{instruction}\n\nTEXT:\n{entry['text']}"},
                {"role": "assistant", "content": json.dumps(entry['interactions'])}
            ]
        })
    return formatted_list

formatted_data = format_for_medgemma(raw_json_list)

View a sample of the formatted data.

In [7]:
formatted_data[0]

{'messages': [{'role': 'user',
   'content': 'You are a medical informatics expert. Extract all drug-drug interactions for the drug \'TENEX\' from the following FDA label text. Return the interactions as a structured JSON list.\n\nTEXT:\nCONTRAINDICATIONS \n\n Tenex is contraindicated in patients with known hypersensitivity to\n guanfacine hydrochloride.PRECAUTIONS \n\n General \n Like other antihypertensive agents, Tenex (guanfacine hydrochloride)\n should be used with caution in patients with severe coronary insufficiency,\n recent myocardial infarction, cerebrovascular disease, or chronic renal or\n hepatic failure. \n\n Sedation \n Tenex, like other orally active central\n α 2 -adrenergic agonists, causes sedation or drowsiness,\n especially when beginning therapy. These symptoms are dose-related (see\n\n ADVERSE REACTIONS \n ). When Tenex is\n used with other centrally active depressants (such as phenothiazines,\n barbiturates, or benzodiazepines), the potential for additive sedat

Convert to a Hugging Face dataset.

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)

print(f"Successfully loaded {len(dataset)} drug labels.")
print("Example Message Format:")
print(dataset[0]['messages'])

Successfully loaded 22 drug labels.
Example Message Format:
[{'content': 'You are a medical informatics expert. Extract all drug-drug interactions for the drug \'TENEX\' from the following FDA label text. Return the interactions as a structured JSON list.\n\nTEXT:\nCONTRAINDICATIONS \n\n Tenex is contraindicated in patients with known hypersensitivity to\n guanfacine hydrochloride.PRECAUTIONS \n\n General \n Like other antihypertensive agents, Tenex (guanfacine hydrochloride)\n should be used with caution in patients with severe coronary insufficiency,\n recent myocardial infarction, cerebrovascular disease, or chronic renal or\n hepatic failure. \n\n Sedation \n Tenex, like other orally active central\n α 2 -adrenergic agonists, causes sedation or drowsiness,\n especially when beginning therapy. These symptoms are dose-related (see\n\n ADVERSE REACTIONS \n ). When Tenex is\n used with other centrally active depressants (such as phenothiazines,\n barbiturates, or benzodiazepines), the 

## Fine-tune the model with LoRA

Traditional fine-tuning of large language models is resource-intensive because it requires adjusting billions of parameters. Parameter-Efficient Fine-Tuning (PEFT) addresses this by training a smaller number of parameters. A common PEFT technique is Low-Rank Adaptation (LoRA), which efficiently adapts large language models by training small, low-rank matrices that are added to the original model instead of updating the full-weight matrices. In QLoRA, the base model is quantized to 4-bit before its weights are frozen, then LoRA adapter layers are attached and trained.

This notebook demonstrates supervised fine-tuning MedGemma with QLoRA using the `SFTTrainer` from the Hugging Face `TRL` library.

### Load model from Hugging Face Hub

Initialize the quantization configuration and load the model.

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# Check if GPU supports bfloat16
if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

### Set up for fine-tuning

Create a [`LoraConfig`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig). It will be provided to the `SFTTrainer`, which supports built-in integration with the Hugging Face `PEFT` library.

In [10]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

Configure training parameters in an [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig).

In [11]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./medgemma-drug-extraction",
    max_length=4096, # Your drug labels are long!
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_steps=100,
    logging_steps=10,
    bf16=True,
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
)

Configure data collator

In [12]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False, # Casual LM, no masking
)

### Fine-tune the model

Construct an [`SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer) using the previously defined LoRA configuration, custom data collator, and training parameters.

In [13]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    data_collator=data_collator
)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Tokenizing train dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

Launch the fine-tuning process.

**Note**: This may take around 3 hours to run using the default configuration.

In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`messages` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

Save the final model to Hugging Face Hub.

In [ ]:
trainer.save_model()

Free up memory before proceeding to evaluate and test the fine-tuned model.

In [ ]:
del model
del trainer
torch.cuda.empty_cache()

## Evaluate the fine-tuned model

### Prepare test dataset

The [CRC-VAL-HE-7K](https://zenodo.org/records/1214456) dataset contains image patches from patients with colorectal adenocarcinoma and does not overlap with NCT-CRC-HE-100K. It can be used as the test dataset to evaluate the fine-tuned MedGemma model.

**Note:** The full CRC-VAL-HE-7K dataset contains over 7K samples. By default this guide only uses a subset with 1,000 samples to keep the evaluation example small.

Download and prepare the test dataset.

In [ ]:
! wget -nc -q "https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip"
! unzip -q CRC-VAL-HE-7K.zip

In [ ]:
from typing import Any

from datasets import load_dataset


def format_test_data(example: dict[str, Any]) -> dict[str, Any]:
    example["messages"] = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": PROMPT,
                },
            ],
        },
    ]
    return example


test_data = load_dataset("./CRC-VAL-HE-7K", split="train")
test_data = test_data.shuffle(seed=42).select(range(1000))
test_data = test_data.map(format_test_data)

### Set up for evaluation

Load the accuracy and F1 score metrics to evaluate the model's performance on the classfication task.

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# Ground-truth labels
REFERENCES = test_data["label"]


def compute_metrics(predictions: list[int]) -> dict[str, float]:
    metrics = {}
    metrics.update(accuracy_metric.compute(
        predictions=predictions,
        references=REFERENCES,
    ))
    metrics.update(f1_metric.compute(
        predictions=predictions,
        references=REFERENCES,
        average="weighted",
    ))
    return metrics

Define a postprocessing function to convert responses to integer class labels before computing metrics.

In [ ]:
from datasets import ClassLabel

# Rename the class names to the tissue classes, `X: tissue type`
test_data = test_data.cast_column(
    "label",
    ClassLabel(names=TISSUE_CLASSES)
)

LABEL_FEATURE = test_data.features["label"]
# Mapping to alternative label format, `(X) tissue type`
ALT_LABELS = dict([
    (label, f"({label.replace(': ', ') ')}") for label in TISSUE_CLASSES
])


def postprocess(prediction: list[dict[str, str]], do_full_match: bool=False) -> int:
    response_text = prediction[0]["generated_text"]
    if do_full_match:
        return LABEL_FEATURE.str2int(response_text)
    for label in TISSUE_CLASSES:
        # Search for `X: tissue type` or `(X) tissue type` in the response
        if label in response_text or ALT_LABELS[label] in response_text:
            return LABEL_FEATURE.str2int(label)
    return -1

### Compute baseline metrics on the pretrained model

Load the pretrained model using the `pipeline` API.

In [ ]:
from transformers import pipeline

pt_pipe = pipeline(
    "image-text-to-text",
    model=model_id,
    torch_dtype=torch.bfloat16,
)

# Set `do_sample = False` for deterministic responses
pt_pipe.model.generation_config.do_sample = False
pt_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

Run batch inference on the test dataset.

In [ ]:
pt_outputs = pt_pipe(
    text=test_data["messages"],
    images=test_data["image"],
    max_new_tokens=40,
    batch_size=64,
    return_full_text=False,
)

pt_predictions = [postprocess(out) for out in pt_outputs]

Compute metrics.

In [ ]:
pt_metrics = compute_metrics(pt_predictions)
print(f"Baseline metrics: {pt_metrics}")

Baseline metrics: {'accuracy': 0.43, 'f1': 0.3520150005688922}


### Compute metrics on the fine-tuned model

Load the base model with the fine-tuned LoRA adapter using the `pipeline` API.

In [ ]:
ft_pipe = pipeline(
    "image-text-to-text",
    model=args.output_dir,
    processor=processor,
    torch_dtype=torch.bfloat16,
)

# Set `do_sample = False` for deterministic responses
ft_pipe.model.generation_config.do_sample = False
ft_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id
# Use left padding during inference
processor.tokenizer.padding_side = "left"

Run batch inference on the test dataset.

In [ ]:
ft_outputs = ft_pipe(
    text=test_data["messages"],
    images=test_data["image"],
    max_new_tokens=20,
    batch_size=64,
    return_full_text=False,
)

ft_predictions = [postprocess(out, do_full_match=True) for out in ft_outputs]

Compute metrics.

In [ ]:
ft_metrics = compute_metrics(ft_predictions)
print(f"Fine-tuned metrics: {ft_metrics}")

Fine-tuned metrics: {'accuracy': 0.945, 'f1': 0.9440669568493043}


# Next steps

Explore the other [notebooks](https://github.com/google-health/medgemma/blob/main/notebooks) to learn what else you can do with the model.